# Treino XGBoost — fogo em [D 09:00, D+7 09:00) UTC por célula H3 res6 (RF10/CA08)

Lê `gold/ml_dataset` (Delta), treina XGBoost e avalia contra baselines em validação e teste cronológicos.

**Decisões aprovadas (dossiê `.omp/tasks/lake-camadas-prata-gold.md`):** emissão D 12:00 UTC com margem de 3 h (entradas até D 09:00);
alvo = detecção FIRMS em [D 09:00, D+7 09:00); `target=0` só com passagem de satélite em todas as 7 janelas; desconhecidos fora;
universo = células res6 de terra já queimadas antes de D; dados completos (sem amostragem) a partir de `EVAL_START`.

**Parâmetros ainda PROPOSTOS, pendentes de aprovação humana:** datas de validação/teste, embargo, `TRAIN_FRACTION`,
hiperparâmetros, `TOP_K`. Nenhum limiar de aceite está definido aqui: o notebook mede, não aprova.

**Limitações:** passagem ≠ visada limpa (nuvem não excluída); cobertura avaliada na célula res4 pai; sensores não harmonizados;
PR-AUC do modelo é calculada com escores arredondados a 5 casas (aproximação); baseline de climatologia é **regional**
(H3 res1 × mês), pois os negativos "quiet" estão agregados por região e não permitem climatologia por célula.

Rodar em Colab: preencha os segredos `AWS_ACCESS_KEY_ID`/`AWS_SECRET_ACCESS_KEY` (e `AWS_SESSION_TOKEN` se temporária) com
uma credencial **somente leitura** do bucket. Local: exporte `ML_DATASET_URI` com o caminho da tabela.

In [ ]:
%pip install -q "deltalake>=1.6,<2" "duckdb==2.0.0" "xgboost>=3,<4" "pyarrow>=19,<24" "h3>=4.2,<5" "numpy>=2,<3"

In [ ]:
from datetime import date, timedelta
import hashlib
import json
import os
from pathlib import Path
import platform
import time

import duckdb
import h3
import numpy as np
import pyarrow.compute as pc
import xgboost as xgb
from deltalake import DeltaTable

try:  # Colab secrets -> environment; never print or store them
    from google.colab import userdata
    for name in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"):
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass
except ImportError:
    pass

## Parâmetros

In [ ]:
DATASET_URI = os.environ.get("ML_DATASET_URI", "s3://data-lake-fire-atlas-553607359608-us-east-2-an/gold/ml_dataset")
STORAGE_OPTIONS = {"AWS_REGION": "us-east-2"} if DATASET_URI.startswith("s3://") else None
OUTPUT_DIR = Path(os.environ.get("TRAIN_OUTPUT_DIR", "train-output"))

# PROPOSTO (pendente de aprovação): blocos cronológicos dentro do período completo do dataset.
EVAL_START = date.fromisoformat(os.environ.get("EVAL_START", "2023-01-01"))       # deve ser o do dataset
VALIDATION_END = date.fromisoformat(os.environ.get("VALIDATION_END", "2024-12-31"))
EMBARGO_DAYS = int(os.environ.get("EMBARGO_DAYS", "7"))                           # >= 7 (horizonte do alvo)
TRAIN_FRACTION = float(os.environ.get("TRAIN_FRACTION", "0.05"))       # fração das linhas `cell` de treino usadas
EARLY_STOP_FRACTION = float(os.environ.get("EARLY_STOP_FRACTION", "0.02"))  # fração da validação p/ early stopping
TOP_K = [int(k) for k in os.environ.get("TOP_K", "100,1000,10000").split(",")]
SAMPLE_SEED = "rf10-ca08-train"
NUM_BOOST_ROUND = int(os.environ.get("NUM_BOOST_ROUND", "2000"))
EARLY_STOPPING_ROUNDS = 50
XGB_PARAMS = {"objective": "binary:logistic", "eval_metric": "aucpr", "tree_method": "hist", "max_depth": 8,
              "eta": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 10, "max_bin": 256,
              "seed": 7, "nthread": os.cpu_count()}

FEATURES = ["det_24h", "det_3d", "det_7d", "frp_mean_7d", "modis_brightness_mean_7d",
            "viirs_brightness_mean_7d", "ring1_det_24h", "ring1_det_7d", "ring2_det_7d"]
TARGET_DAYS = 7
assert EMBARGO_DAYS >= TARGET_DAYS, "embargo menor que o horizonte vaza alvo entre blocos"
assert 0 < TRAIN_FRACTION <= 1 and 0 < EARLY_STOP_FRACTION <= 1

## Dataset: versão, metadados e blocos

In [ ]:
table = DeltaTable(DATASET_URI, storage_options=STORAGE_OPTIONS)
commits = [c for c in table.history() if "forecast_month" in c]
for key, expected in (("eval_start", EVAL_START.isoformat()), ("cutoff_hour", "9")):
    found = {c[key] for c in commits}
    assert found == {expected}, f"dataset {key}={found}, notebook espera {expected}"
months = sorted({(int(p["year"]), int(p["month"])) for p in table.partitions()})
con = duckdb.connect()
con.execute("SET TimeZone='UTC'")
con.register("ml", table.to_pyarrow_dataset())
first_d, last_d = con.execute("SELECT min(forecast_date), max(forecast_date) FROM ml").fetchone()

BLOCKS = {  # D inclusive; embargo entre blocos impede que janelas-alvo cruzem para o bloco seguinte
    "train": (first_d, EVAL_START - timedelta(days=EMBARGO_DAYS)),
    "validation": (EVAL_START, VALIDATION_END - timedelta(days=EMBARGO_DAYS)),
    "test": (VALIDATION_END + timedelta(days=1), last_d),
}
for name, (lo, hi) in BLOCKS.items():
    assert lo <= hi, f"bloco {name} vazio: {lo}..{hi}"

def block_months(name):
    lo, hi = BLOCKS[name]
    return [(y, m) for y, m in months if date(y, m, 1) <= hi and (date(y, m, 28) + timedelta(days=4)).replace(day=1) > lo]

print("versão do dataset:", table.version(), "| meses:", len(months), f"| D {first_d}..{last_d}")
print({k: (str(lo), str(hi), len(block_months(k))) for k, (lo, hi) in BLOCKS.items()})

## Leitura por mês (memória limitada)

In [ ]:
def month_table(name, year, month, fraction=None):
    """Linhas rotuladas (peso > 0) de um mês dentro do bloco; com `fraction`, sorteio md5 determinístico
    das linhas `cell` e peso / fraction (agregados `quiet` entram sempre, já exatos)."""
    lo, hi = BLOCKS[name]
    keep, scale = "TRUE", "1.0"
    if fraction is not None and fraction < 1:
        drawn = f"md5_number(concat(h3_r6::VARCHAR, '|', forecast_date::VARCHAR, '|', '{SAMPLE_SEED}')) % 1000000"
        keep = f"(row_kind <> 'cell' OR {drawn} < {round(fraction * 1_000_000)})"
        scale = f"CASE WHEN row_kind = 'cell' THEN {1 / fraction} ELSE 1.0 END"
    return con.execute(f"""
        SELECT {', '.join(FEATURES)}, target::DOUBLE AS target, weight * {scale} AS w,
               forecast_date, h3_r1, row_kind
        FROM ml
        WHERE year = {year} AND month = {month} AND target IS NOT NULL AND weight > 0
          AND forecast_date BETWEEN DATE '{lo}' AND DATE '{hi}' AND {keep}
    """).arrow()

def matrix(tbl):
    return np.column_stack([pc.cast(tbl[f], "float32").to_numpy(zero_copy_only=False) for f in FEATURES])

class MonthIter(xgb.DataIter):
    def __init__(self, name, fraction):
        self.name, self.fraction, self.months, self.i = name, fraction, block_months(name), 0
        super().__init__()
    def next(self, input_data):
        while self.i < len(self.months):
            tbl = month_table(self.name, *self.months[self.i], fraction=self.fraction)
            self.i += 1
            if tbl.num_rows:
                input_data(data=matrix(tbl), label=tbl["target"].to_numpy(), weight=tbl["w"].to_numpy())
                return True
        return False
    def reset(self):
        self.i = 0

## Treino

In [ ]:
started = time.time()
dtrain = xgb.QuantileDMatrix(MonthIter("train", TRAIN_FRACTION), max_bin=XGB_PARAMS["max_bin"])
dstop = xgb.QuantileDMatrix(MonthIter("validation", EARLY_STOP_FRACTION), ref=dtrain)
print(f"treino: {dtrain.num_row():,} linhas | early stopping: {dstop.num_row():,} linhas")
booster = xgb.train(XGB_PARAMS, dtrain, num_boost_round=NUM_BOOST_ROUND, evals=[(dstop, "validation_sample")],
                    early_stopping_rounds=EARLY_STOPPING_ROUNDS, verbose_eval=100)
print("melhor iteração:", booster.best_iteration, f"| {time.time() - started:.0f} s")

## Baselines
- **Persistência:** escore = `det_7d` (detecções da própria célula nos 7 dias anteriores ao corte).
- **Climatologia regional:** taxa ponderada de positivos por (H3 res1, mês do ano) no bloco de treino (todas as linhas, sem amostragem).

In [ ]:
climatology = {}
for year, month in block_months("train"):
    rows = con.execute(f"""
        SELECT h3_r1, month(forecast_date) AS moy, sum(weight * target), sum(weight) FROM ml
        WHERE year = {year} AND month = {month} AND target IS NOT NULL AND weight > 0
          AND forecast_date <= DATE '{BLOCKS["train"][1]}'
        GROUP BY 1, 2""").fetchall()
    for r1, moy, pos, total in rows:
        acc = climatology.setdefault((r1, moy), [0.0, 0.0])
        acc[0] += pos
        acc[1] += total
global_rate = sum(a for a, _ in climatology.values()) / sum(b for _, b in climatology.values())
climatology = {k: a / b for k, (a, b) in climatology.items() if b > 0}

def climatology_score(tbl):
    moy = pc.month(tbl["forecast_date"]).to_numpy()
    r1 = tbl["h3_r1"].to_numpy()
    return np.array([climatology.get((c, m), global_rate) for c, m in zip(r1, moy)], dtype=np.float64)
print("células-região na climatologia:", len(climatology), "| taxa global:", round(global_rate, 5))

## Avaliação completa (sem amostragem) em validação e teste
Métricas ponderadas por `w` (agregados `quiet` contam cada célula). Top-k é por dia de emissão, sobre células.

In [ ]:
class Metrics:
    """Acumula, mês a mês, PR-AUC (average precision), Brier, calibração e precisão/recall@k diários."""
    def __init__(self, probabilistic):
        self.probabilistic, self.pos, self.neg = probabilistic, {}, {}
        self.brier, self.weight, self.calibration, self.daily = 0.0, 0.0, np.zeros((10, 3)), []

    def add(self, score, target, weight, day):
        key = np.round(score, 5)
        for book, mask in ((self.pos, target == 1), (self.neg, target == 0)):
            values, inverse = np.unique(key[mask], return_inverse=True)
            for value, total in zip(values, np.bincount(inverse, weights=weight[mask])):
                book[value] = book.get(value, 0.0) + total
        self.weight += weight.sum()
        if self.probabilistic:
            self.brier += (weight * (score - target) ** 2).sum()
            bins = np.minimum((score * 10).astype(int), 9)
            np.add.at(self.calibration, bins, np.column_stack([weight, weight * score, weight * target]))
        for d in np.unique(day):
            m = day == d
            order = np.argsort(-score[m], kind="stable")
            w, y = weight[m][order], target[m][order]
            cum_w, cum_pos = np.cumsum(w), np.cumsum(w * y)
            positives = cum_pos[-1]
            row = {}
            for k in TOP_K:
                i = np.searchsorted(cum_w, k)  # linha onde o k-ésimo cell cai (agregado pode ser cortado)
                if i >= len(w):
                    hit = positives
                else:
                    before_w = cum_w[i - 1] if i else 0.0
                    before_p = cum_pos[i - 1] if i else 0.0
                    hit = before_p + (k - before_w) * y[i]
                row[k] = (hit / k, hit / positives if positives else np.nan)
            self.daily.append(row)

    def result(self):
        values = sorted(set(self.pos) | set(self.neg), reverse=True)
        tp = np.cumsum([self.pos.get(v, 0.0) for v in values])
        fp = np.cumsum([self.neg.get(v, 0.0) for v in values])
        precision, recall = tp / (tp + fp), tp / tp[-1]
        ap = float(np.sum(np.diff(np.concatenate([[0.0], recall])) * precision))
        out = {"pr_auc": ap, "positive_rate": float(tp[-1] / (tp[-1] + fp[-1])), "weighted_cells": float(self.weight)}
        if self.probabilistic:
            out["brier"] = float(self.brier / self.weight)
            out["calibration"] = [{"bin": f"{b/10:.1f}-{(b+1)/10:.1f}", "weight": float(c[0]),
                                   "mean_score": float(c[1] / c[0]) if c[0] else None,
                                   "observed_rate": float(c[2] / c[0]) if c[0] else None}
                                  for b, c in enumerate(self.calibration)]
        for k in TOP_K:
            out[f"precision@{k}"] = float(np.nanmean([r[k][0] for r in self.daily]))
            out[f"recall@{k}"] = float(np.nanmean([r[k][1] for r in self.daily]))
        return out

hemisphere = {}
def hemisphere_of(r1):
    if r1 not in hemisphere:
        hemisphere[r1] = "N" if h3.cell_to_latlng(h3.int_to_str(int(r1)))[0] >= 0 else "S"
    return hemisphere[r1]

MODELS = {"xgboost": True, "persistence": False, "regional_climatology": True}
report = {}
for split in ("validation", "test"):
    books = {(model, part): Metrics(prob) for model, prob in MODELS.items() for part in ("all", "N", "S")}
    for year, month in block_months(split):
        tbl = month_table(split, year, month)
        if not tbl.num_rows:
            continue
        X = matrix(tbl)
        scores = {"xgboost": booster.predict(xgb.DMatrix(X), iteration_range=(0, booster.best_iteration + 1)),
                  "persistence": tbl["det_7d"].to_numpy().astype(np.float64),
                  "regional_climatology": climatology_score(tbl)}
        target, weight = tbl["target"].to_numpy(), tbl["w"].to_numpy()
        day = tbl["forecast_date"].to_numpy()
        half = np.array([hemisphere_of(r) for r in tbl["h3_r1"].to_numpy()])
        for model, score in scores.items():
            for part in ("all", "N", "S"):
                m = slice(None) if part == "all" else half == part
                books[(model, part)].add(score[m], target[m], weight[m], day[m])
        print(split, f"{year}-{month:02d}", f"{tbl.num_rows:,} linhas")
    report[split] = {model: {part: books[(model, part)].result() for part in ("all", "N", "S")} for model in MODELS}

for split, models in report.items():
    print(f"\n{split}")
    for model, parts in models.items():
        a = parts["all"]
        print(f"  {model:22s} PR-AUC {a['pr_auc']:.4f}  Brier {a.get('brier', float('nan')):.5f}  "
              + "  ".join(f"P@{k} {a[f'precision@{k}']:.3f}" for k in TOP_K))

## Artefatos

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
booster.save_model(OUTPUT_DIR / "model.json")
model_sha256 = hashlib.sha256((OUTPUT_DIR / "model.json").read_bytes()).hexdigest()
run = {
    "status": "experimento; limiares de aceite não definidos — não é aprovação do modelo",
    "dataset": {"uri": DATASET_URI, "delta_version": table.version(), "forecast_dates": [str(first_d), str(last_d)]},
    "blocks": {k: [str(lo), str(hi)] for k, (lo, hi) in BLOCKS.items()},
    "params": {"embargo_days": EMBARGO_DAYS, "train_fraction": TRAIN_FRACTION,
               "early_stop_fraction": EARLY_STOP_FRACTION, "sample_seed": SAMPLE_SEED, "top_k": TOP_K,
               "xgb": XGB_PARAMS, "best_iteration": booster.best_iteration, "features": FEATURES},
    "rows": {"train": dtrain.num_row(), "early_stopping": dstop.num_row()},
    "feature_gain": booster.get_score(importance_type="gain"),
    "metrics": report,
    "model_sha256": model_sha256,
    "environment": {"python": platform.python_version(), "xgboost": xgb.__version__, "duckdb": duckdb.__version__},
}
(OUTPUT_DIR / "run.json").write_text(json.dumps(run, indent=2, default=str))
print("gravado em", OUTPUT_DIR.resolve(), "| model sha256", model_sha256[:12])